# Interactive map

Goal: Implement an interactive map that allows filtering of projects by type (funding_scheme), by total cost, by type, by field / subfield / niche, startyear of the project.

We create several interactive maps with plotly:
- Bubble chart: country-specific bubbble
    - Bubble magnitude needs a metric
    - Metric: total contribution / total cost / number of projects (normalized by number of inhabitants) / number of publications /100k money received
    - Add colormap right to the plot that colorcodes the encoded metric
    - Allow user to click on bubble and then zoom in one layer, on the country user clicked on. Then recreate the same map, but this time on the level of cities. 
Requirements:
- `plotly`: creating interactive plots
- `pandas`: data management
- `numpy`: 
- `folium`

In [14]:
# imports
import numpy as np
import pandas as pd
import plotly.express as px
import os
import sys
from pathlib import Path

from ipywidgets import interact, widgets
from IPython.display import display, clear_output

## Load data

In [9]:
project_root = Path.cwd().parent  # assumes you're in /notebooks
sys.path.append(str(project_root))

from backend.etl.ingestion import inspect_bad_lines, auto_fix_row, robust_csv_reader

In [27]:
# get projects and organizations
notebook_dir = os.getcwd()
base_dir = os.path.dirname(notebook_dir)

df_proj_processed = robust_csv_reader(f'{base_dir}/data/processed/project_df.csv', delimiter=',')
df_proj_interim = robust_csv_reader(f'{base_dir}/data/interim/project_df.csv', delimiter=',')
df_org = robust_csv_reader(f'{base_dir}/data/processed/organization_df.csv', delimiter=',')
df_topics = robust_csv_reader(f'{base_dir}/data/processed/topics_df.csv', delimiter=',')

In [34]:
list(df_proj_processed.keys())

['id',
 'acronym',
 'status',
 'title',
 'startDate',
 'endDate',
 'totalCost',
 'ecMaxContribution',
 'legalBasis',
 'topics',
 'ecSignatureDate',
 'frameworkProgramme',
 'masterCall',
 'subCall',
 'fundingScheme',
 'nature',
 'objective',
 'contentUpdateDate',
 'rcn',
 'grantDoi',
 'duration_days',
 'duration_months',
 'duration_years',
 'projectID_x',
 'n_institutions',
 'projectID_y',
 'institutions',
 'projectID',
 'coordinator_name',
 'ecContribution_per_year',
 'totalCost_per_year',
 'field_class',
 'field',
 'subfield',
 'niche']

In [36]:
# change the keys:
columns_renamed = {
    'status': 'status',
    'title': 'title',
    'startDate': 'start_date',
    'endDate': 'end_date',
    'totalCost': 'total_cost',
    'ecMaxContribution': 'ec_max_contribution',
    'ecSignatureDate': 'ec_signature_date',
    'frameworkProgramme': 'framework_programme',
    'masterCall': 'master_call',
    'subCall': 'sub_call',
    'fundingScheme': 'funding_scheme',
    'nature': 'nature',
    'objective': 'objective',
    'contentUpdateDate': 'content_update_date',
    'rcn': 'rcn',
    'grantDoi': 'grant_doi',
    'duration_days': 'duration_days',
    'duration_months': 'duration_months',
    'duration_years': 'duration_years',
    'n_institutions': 'n_institutions',
    'coordinator_name': 'coordinator_name',
    'ecContribution_per_year': 'ec_contribution_per_year',
    'totalCost_per_year': 'total_cost_per_year',
    'field_class': 'field_class',
    'field': 'field',
    'subfield': 'sub_field',
    'niche': 'niche'
}

df_proj = df_proj.rename(columns=columns_renamed)

In [37]:
df_proj

,id,acronym,status,title,start_date,end_date,total_cost,ec_max_contribution,legalBasis,topics,...,n_institutions,projectID_y,institutions,projectID,coordinator_name,ec_contribution_per_year,total_cost_per_year,sci_voc_titles,sci_voc_paths,topic_titles
0,101159220,PvSeroRDT,SIGNED,A point-of-care serological rapid diagnostic t...,2025-02-01,2030-01-31,,,HORIZON.2.1,HORIZON-JU-GH-EDCTP3-2023-02-02-two-stage,...,8,101159220,"['Institut Pasteur de Madagascar', 'INSTITUT P...",101159220,INSTITUT PASTEUR,,,"['malaria', 'proteins']",['/medical and health sciences/health sciences...,['Advancing point-of-care diagnostics to the m...
1,101096150,BIOBoost,SIGNED,Boosting innovation agencies for bioeconomy va...,2023-02-01,2025-01-31,0.0,500000.0,HORIZON.3.2,HORIZON-EIE-2022-CONNECT-01-01,...,8,101096150,"['CLIC INNOVATION OY', 'FBCD AS', 'BIOECONOMY ...",101096150,FBCD AS,500000.0,0.0,,,['Towards more inclusive networks and initiati...
2,101093997,GlycanTrigger,SIGNED,GLYCANS AS MASTER TRIGGERS OF HEALTH TO INTEST...,2023-01-01,2028-12-31,6771571.0,6771571.0,HORIZON.2.1,HORIZON-HLTH-2022-STAYHLTH-02-01,...,10,101093997,"['ACADEMISCH ZIEKENHUIS LEIDEN', 'LUDGER LIMIT...",101093997,I3S - INSTITUTO DE INVESTIGACAO E INOVACAO EM ...,1354314.2,1354314.2,['carbohydrates'],['/natural sciences/biological sciences/bioche...,['Personalised blueprint of chronic inflammati...
3,101126531,CHIKVAX_CHIM,SIGNED,Late-stage clinical development of Chikungunya...,2023-06-01,2028-11-30,100000000.0,70000000.0,HORIZON.2.1,HORIZON-HLTH-2022-CEPI-15-01-IBA,...,1,101126531,['COALITION FOR EPIDEMIC PREPAREDNESS INNOVATI...,101126531,COALITION FOR EPIDEMIC PREPAREDNESS INNOVATIONS,14000000.0,20000000.0,"['virology', 'coronaviruses', 'vaccines']",['/natural sciences/biological sciences/microb...,['CEPI 4 - Contribution to the Coalition for E...
4,101113979,The Oater,CLOSED,The Oater develops a compact machine for hyper...,2023-07-01,2023-12-31,0.0,75000.0,HORIZON.3.2,HORIZON-EIE-2022-SCALEUP-02-02,...,1,101113979,['OIY SOLUTIONS GMBH'],101113979,OIY SOLUTIONS GMBH,inf,,"['internet of things', 'e-commerce', 'ecosyste...",['/natural sciences/computer and information s...,['Women TechEU']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15858,101052410,EUCYS2022,CLOSED,EUCYS Leiden2022,2021-09-01,2023-02-28,2000000.0,2000000.0,HORIZON.4.2,HORIZON-WIDERA-2021-EUCYS-IBA,...,1,101052410,['STICHTING LEIDEN EUROPEAN CITY OF SCIENCE 20...,101052410,STICHTING LEIDEN EUROPEAN CITY OF SCIENCE 2022,2000000.0,2000000.0,,,['European Union Contest for Young Scientists ...
15859,101124648,RESAVER_2023,SIGNED,Support to Retirement Savings Vehicle for Euro...,2023-09-01,2026-08-31,,,HORIZON.4.2,HORIZON-WIDERA-2023-RESAVER-IBA,...,1,101124648,['RETIREMENT SAVINGS VEHICLE FOR EUROPEAN RESE...,101124648,RETIREMENT SAVINGS VEHICLE FOR EUROPEAN RESEAR...,,,,,['Support to Retirement Savings Vehicle for Eu...
15860,101052247,Leiden2022-ECS-ESOF,CLOSED,European City of Science and EuroScience Open ...,2021-08-01,2023-03-31,,2000000.0,HORIZON.4.2,HORIZON-WIDERA-2021-ESOF-IBA,...,1,101052247,['STICHTING LEIDEN EUROPEAN CITY OF SCIENCE 20...,101052247,STICHTING LEIDEN EUROPEAN CITY OF SCIENCE 2022,2000000.0,,,,['The EuroScience Open Forum (ESOF) and Europe...
15861,101172981,EUCYS2024,SIGNED,European Union Contest for Young Scientists (E...,2024-02-01,2025-02-28,999500.0,999500.0,HORIZON.4.2,HORIZON-WIDERA-2024-EUCYS-IBA,...,1,101172981,['UNIWERSYTET SLASKI W KATOWICACH'],101172981,UNIWERSYTET SLASKI W KATOWICACH,999500.0,999500.0,,,['European Union Contest for Young Scientists ...


## Start creating the plot
We implement the features implemented above. 

In [15]:
df_org['country'].unique()

array(['MG', 'SN', 'ET', 'UK', 'CH', 'FR', 'AU', 'FI', 'DK', 'ES', 'SI',
       'LT', 'PL', 'NL', 'PT', 'BE', 'DE', 'US', 'NO', 'TR', 'ZA', 'ZM',
       'ZW', 'CI', 'SK', 'BG', 'RO', 'EL', 'IL', 'IT', 'EE', 'IE', 'HU',
       'CZ', 'AT', 'LV', 'UA', 'GN', 'ML', 'SE', 'BW', 'MZ', 'LS', '',
       'SZ', 'BF', 'GH', 'CY', 'MT', 'CM', 'LU', 'NG', 'TZ', 'MW', 'UG',
       'KE', 'CN', 'IN', 'KR', 'RS', 'EG', 'AR', 'HR', 'AM', 'BR', 'CV',
       'CA', 'TN', 'AO', 'ST', 'CO', 'BT', 'PY', 'CF', 'DZ', 'GQ', 'LK',
       'CL', 'AL', 'IS', 'CD', 'BI', 'MX', 'ME', 'MN', 'TH', 'KZ', 'JP',
       'VA', 'NZ', 'EC', 'MD', 'UZ', 'AZ', 'SG', 'PK', 'TW', 'GU', 'CR',
       'PE', 'LB', 'BA', 'MA', 'VN', 'MK', 'BJ', 'GA', 'MY', 'XK', 'PS',
       'PH', 'SA', 'RW', 'ID', 'FO', 'CU', 'KG', 'BD', 'PF', 'LR', 'SL',
       'VE', 'GE', 'JO', 'FJ', 'UY', 'CG', 'AF', 'IQ', 'HK', 'TJ', 'TM',
       'BO', 'MV', 'IM', 'NP', 'MH', 'AD', 'MU', 'PA', 'DJ', 'TD', 'BQ',
       'AW', 'GM', 'MR', 'TG', 'SD', 'PG', 'LA', 'MO'

Create dictionary with country-specific population numbers. Generated by ChatGPT so take care with those values

In [38]:
country_populations = {
    'MG': 29452714, 'SN': 18847519, 'ET': 118550298, 'UK': 68459055, 'CH': 8860574, 'FR': 68374591,
    'AU': 26768598, 'FI': 5626414, 'DK': 5973136, 'ES': 47280433, 'SI': 2097893, 'LT': 2628186,
    'PL': 38746310, 'NL': 17772378, 'PT': 10207177, 'BE': 11977634, 'DE': 84119100, 'US': 347275807,
    'NO': 5509733, 'TR': 84119531, 'ZA': 60442647, 'ZM': 20799116, 'ZW': 17150352, 'CI': 29981758,
    'SK': 5563649, 'BG': 6782659, 'RO': 18148155, 'EL': 10461091, 'IL': 9402617, 'IT': 60964931,
    'EE': 1193791, 'IE': 5233461, 'HU': 9855745, 'CZ': 10837890, 'AT': 8967982, 'LV': 1801246,
    'UA': 35661826, 'GN': 13986179, 'ML': 21990607, 'SE': 10589835, 'BW': 2450668, 'MZ': 33350954,
    'LS': 2227548, 'SZ': 1138089, 'BF': 23042199, 'GH': 34589092, 'CY': 1320525, 'MT': 469730,
    'CM': 30966105, 'LU': 671254, 'NG': 237527782, 'TZ': 67462121, 'MW': 21763309, 'UG': 49283041,
    'KE': 58246378, 'CN': 1416096094, 'IN': 1463865525, 'KR': 52081799, 'RS': 6652212,
    'EG': 111247248, 'AR': 46994384, 'HR': 4150116, 'AM': 2976765, 'BR': 212812405, 'CV': 611014,
    'CA': 38794813, 'TN': 12048847, 'AO': 37202061, 'ST': 223561, 'CO': 49588357, 'BT': 884546,
    'PY': 7522549, 'CF': 5650957, 'DZ': 47022473, 'GQ': 1795834, 'LK': 21982608, 'CL': 18664652,
    'AL': 3107100, 'IS': 364036, 'CD': 115403027, 'BI': 13590102, 'MX': 130739927, 'ME': 599849,
    'MN': 3281676, 'TH': 69920998, 'KZ': 20260006, 'JP': 123201945, 'VA': 496, 'NZ': 5161211,
    'EC': 18309984, 'MD': 3599528, 'UZ': 36520593, 'AZ': 10650239, 'SG': 6028459, 'PK': 255219554,
    'TW': 23595274, 'GU': 169532, 'CR': 5265575, 'PE': 32600249, 'LB': 5364482, 'BA': 3798671,
    'MA': 37387585, 'VN': 105758975, 'MK': 2135622, 'BJ': 14697052, 'GA': 2455105, 'MY': 34564810,
    'XK': 1977093, 'PS': 3243369, 'PH': 118277063, 'SA': 36544431, 'RW': 13623302, 'ID': 285721236,
    'FO': 52933, 'CU': 10966038, 'KG': 6172101, 'BD': 175686899, 'PF': 303540, 'LR': 5437249,
    'SL': 9121049, 'VE': 31250306, 'GE': 4900961, 'JO': 11174024, 'FJ': 951611, 'UY': 3425330,
    'CG': 6097665, 'AF': 40121552, 'IQ': 42083436, 'HK': 7297821, 'TJ': 10394063, 'TM': 5744151,
    'BO': 12311974, 'MV': 388858, 'IM': 92269, 'NP': 31122387, 'MH': 82011, 'AD': 85370,
    'MU': 1310504, 'PA': 4470241, 'DJ': 994974, 'TD': 19093595, 'BQ': 25519, 'AW': 125063,
    'GM': 2523327, 'MR': 4328040, 'TG': 8917994, 'SD': 50467278,
    '': None  # No country code provided
}


In [16]:
# Filter coordinators only
df_coordinators = df_proj_org[df_proj_org['role'].str.lower() == 'coordinator']
df_coordinators = df_coordinators.merge(df_org[['id', 'country']], left_on='organization_id', right_on='id')
df_coordinators = df_coordinators.rename(columns={'project_id': 'id'})

# Merge minimal fields from project table (no full join!)
df_summary = df_coordinators.merge(df_proj[[
    'id', 'total_cost', 'ec_max_contribution', 'field', 'sub_field', 'niche',
    'funding_scheme', 'start_year'
]], on='id')

NameError: name 'df_proj_org' is not defined

In [ ]:
# add interactive options
interact(
    plot_filtered_map,
    metric = widgets.Dropdown(options=[
        'Total Contribution', 'Total Cost', 'Projects per Country', '€/100k inhabitants'
    ]),
    funding_scheme = widgets.Dropdown(
        options=[None] + sorted(df_proj['funding_scheme'].dropna().unique())
    ),
    field = widgets.Dropdown(
        options=[None] + sorted(df_proj['field'].dropna().unique())
    ),
    sub_field = widgets.Dropdown(
        options=[None] + sorted(df_proj['sub_field'].dropna().unique())
                                ),
    niche = widgets.Dropdown(
        options=[None] + sorted(df_proj['niche'].dropna().unique())
                            ),
    min_total_cost = widgets.FloatSlider(.    # TODO: add min and max based on dataFrame information
        min=0, max=1e7, step=1e5, value=0
                                        ),
    max_total_cost = widgets.FloatSlider(    # TODO: add min and max based on dataFrame information
        min=1e6, max=1e9, step=1e7, value=1e9
                                        ),
    start_year = widgets.IntRangeSlider(     # TODO: add min and max based on dataFrame information    
        min=2014, max=2025, value=(2021, 2025), step=1
                                       )
);